<a href="https://colab.research.google.com/github/donlap/ds352-labs/blob/main/Lab02_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Statistical Learning for Data Science 2 (229352)
#### Instructor: Donlapark Ponnoprat

#### [Course website](https://donlapark.pages.dev/229352/)

## Lab #2

In [ ]:
!pip install -q tabpfn tabicl pytabkit

import os
os.environ["TABPFN_MODEL_VERSION"] = "v2"  # TabPFN-2 does not need an account

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.metrics import root_mean_squared_error

# Black-box models for Exercise 1
from tabpfn import TabPFNRegressor
from tabicl import TabICLRegressor
from pytabkit import RealMLP_TD_Regressor

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Part 1: Steel Industry Energy Dataset - Manual Data Preprocessing

### Load the Steel Industry Energy Dataset ([Data Information](https://archive.ics.uci.edu/dataset/851/steel+industry+energy+consumption))

The data is related with the electricity consumption of DAEWOO Steel Co. Ltd in Gwangyang, South Korea, in 2018. The original data has one row every 15 minutes; here, each row is one hour, and we use the last 50 days of 2018. The column `Usage_kWh` is the electricity used in the hour, and `Load_Type` is the load level of the hour (`'Light_Load'`, `'Medium_Load'` or `'Maximum_Load'`).

In [ ]:
steel_url = 'https://raw.githubusercontent.com/donlap/ds352-labs/main/steel_industry_hourly.csv'

df = pd.read_csv(steel_url, parse_dates=['date'])
df = df[df['date'] >= '2018-11-12'].copy()  # the last 50 days, so that the models train quickly
df['hour'] = df['date'].dt.hour
print("Shape of the dataset:", df.shape)
df.head()

### Data Exploration

In [ ]:
print("--- Missing Values Count ---")
print(df.isnull().sum())

In [ ]:
print("--- Unique Values for Categorical Columns ---")
for col in df.select_dtypes(include='object').columns:
    print(f"\n'{col}' unique values:")
    print(df[col].value_counts(dropna=False)) # Include NaN counts

### Data Preprocessing

In [ ]:
# Sliding window: the usage of the last 3 hours
df['Usage_lag1'] = # Write your code here


# Drop the first 3 rows, which have no past usage


# Drop 'CO2(tCO2)' due to data leakage (it is the usage times a constant)


# Define features (X) and target (y = 'Usage_kWh')


# Split the data BEFORE any transformations (time series: shuffle=False)


# Print data shape



We will apply `StandardScaler()`, `OrdinalEncoder()`, `OneHotEncoder()`, cyclical encoding and squaring on a few selected columns.

**1. Numerical Feature: `Usage_lag1` and `Lagging_Current_Reactive.Power_kVarh` (Standard Scaling)**

In [ ]:
num_cols_demo = ['Usage_lag1', 'Lagging_Current_Reactive.Power_kVarh']

scaler = StandardScaler()

# Fit the scaler ONLY on the training data



**2. Ordinal Feature: `Load_Type` (Ordinal Encoding)**

In [ ]:
load_categories = ['Light_Load', 'Medium_Load', 'Maximum_Load']

In [ ]:
ord_col_demo = ['Load_Type']

ordinal_encoder = OrdinalEncoder(categories=[load_categories])

## Write your code here



**3. Nominal Feature: `Day_of_week` (One-Hot Encoding)**

In [ ]:
nom_col_demo = ['Day_of_week']

onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

## Write your code here



**4. Cyclical Feature: `hour` (Cyclical Encoding)**

$$
\text{hour} \longmapsto \left(\sin\frac{2\pi \cdot \text{hour}}{24},\ \cos\frac{2\pi \cdot \text{hour}}{24}\right)
$$

In [ ]:
cyc_col_demo = 'hour'

## Write your code here



**5. Squared Feature: `Lagging_Current_Reactive.Power_kVarh` (Squaring)**

In [ ]:
sq_col_demo = 'Lagging_Current_Reactive.Power_kVarh'

## Write your code here



### Train a Black-Box Model

**1. Training and Test Sets with All Preprocessed Features (Standard Scaling)**

In [ ]:
# Put the preprocessed features of the training set together, and the same for the test set
## Write your code here


# Fit the scaler ONLY on the training data


# Print data shape



**2. Black-Box Model: `TabPFNRegressor` and the RMSE** ([Documentation](https://github.com/PriorLabs/TabPFN))

In [ ]:
model_demo = TabPFNRegressor()

## Write your code here



## Part 2: Bike Sharing Dataset - Preprocessing & Black-Box Models

### Load the Bike Sharing Dataset ([Data Information](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset))

The data contains the number of bikes rented each day (`cnt`) from Capital Bikeshare in Washington, D.C., in 2011 and 2012, together with the weather of each day. The columns `temp`, `hum` and `windspeed` are already scaled to values between 0 and 1 by the authors of the dataset. The column `weather` is one of `'clear'`, `'mist'`, `'light rain/snow'` and `'heavy rain/snow'`.

In [ ]:
bike_url = 'https://raw.githubusercontent.com/donlap/ds352-labs/main/bike_sharing_daily.csv'

bike = pd.read_csv(bike_url, parse_dates=['date'])
print("Shape of the dataset:", bike.shape)
bike.head()

### **Exercise 1: Apply All Preprocessing & Train Three Black-Box Models**

Now, it's your turn to apply these preprocessing steps to the Bike Sharing dataset and then train three black-box models that predict `cnt`.

**Instructions:**

1. Look at the Variable Table in [this link](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), and at the columns of `bike` above.
2. **Sliding window:** add the rentals of the last 3 days as new columns `cnt_lag1`, `cnt_lag2` and `cnt_lag3` using [`.shift()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.shift.html). Then drop the first 3 rows.
3. **Squaring:** add a new column `temp_squared`. People rent more bikes on warmer days, but fewer on the hottest days.
4. Define `X` (drop `date` and `cnt`) and `y` (`cnt`). Split the data with `train_test_split(..., test_size=0.2, shuffle=False)`, so that the test set is the last days.
5. Make lists for `numerical_features`, `ordinal_features`, `nominal_features` and `cyclical_features`.
6. Preprocess the features. It is safer to make a copy of `X_train` using:
   ```
   X_train_copy = X_train.copy()
   X_test_copy = X_test.copy()
   ```
   and preprocess `X_train_copy` instead.
    *   Ordinal feature: `weather`, with the order `['clear', 'mist', 'light rain/snow', 'heavy rain/snow']`.
    *   Cyclical feature: `month`, with $\sin(2\pi \cdot \text{month}/12)$ and $\cos(2\pi \cdot \text{month}/12)$.
    *   Numerical features: standard scaling.

   **For nominal features, concat the one-hot encoded features using [`pd.concat(..., axis=1)`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) and drop the old nominal features from the dataframe.**
7. Train three black-box models on the preprocessed `X_train_copy` and `y_train`. They have the same `fit()` and `predict()` as scikit-learn models.
    *   `TabPFNRegressor()` ([Documentation](https://github.com/PriorLabs/TabPFN))
    *   `TabICLRegressor()` ([Documentation](https://github.com/soda-inria/tabicl))
    *   `RealMLP_TD_Regressor(random_state=0)` ([Documentation](https://github.com/dholzmueller/pytabkit))
8. Evaluate the models:
    *   Make predictions on the preprocessed `X_test_copy`.
    *   Print `root_mean_squared_error` ([Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)). Which model has the smallest error?
9. Try leaving out one preprocessing step at a time, for example the sliding window or `temp_squared`. How does the error of each model change?


In [ ]:
# --- YOUR CODE FOR EXERCISE 1 STARTS HERE ---


